# NIG-TimeNet (PyTorch): بناء بيانات التدريب

**Runtime ▸ Run all.** الخلية الوحيدة التي فيها إعدادات هي التالية؛ كل خيار من خيارات خط البيانات مفتاح منقَّط في `CHANGES` (انظر `nig_timenet_torch/data/settings.py`، و`python -m nig_timenet_torch.data.cli show` يطبعها كلها).
الدفتر **مُشغِّل رفيع**: كل المنطق في الحزمة `nig_timenet_torch.data` ولها اختبارات تقارن النوافذ والأهداف والمقاييس **بت ببت** مع مشروع TensorFlow (`tests/data/test_parity_tf.py`).

### المدخلات
الشموع الخام نفسها التي يستعملها `crypto-signal-prediction`: `<RAW_ROOT>/history_1h/<SYMBOL>.csv(.gz)` وسجل الأصول `<RAW_ROOT>/crypto_data/asset_registry.csv` (على Drive).

### المخرجات
مجلد `<NAME>_<الختم الزمني>.dataset` ونسخة `<NAME>_latest.dataset` في `OUT_DIR`. تقرؤه دفاتر التدريب باسمه الأساسي `NAME` (نسخة `_latest`) أو باسمه الكامل المؤرَّخ.
هذا الملف هو «العقد» (artifact، القاعدة D1): يحمل `manifest.json` فيه مصدره وقائمة العملات والعملات المحجوبة المستبعدة وأحدث شمعة (يجب أن تسبق ختم الزمن) وبصمات الملفات، ولا يقبله محمِّل التدريب بدونها.

### الترتيب
1. **Setup:** ربط Drive، استنساخ الكود، تثبيت الحزم، التحقق من الجهاز.
2. **Settings:** `PRESETS` ثم `CHANGES` فوقها؛ مفتاح خاطئ يفشل باقتراح قبل بناء أي شيء.
3. **Download (اختياري):** تحديث الشموع الخام بأحدث شموع الساعة.
4. **Build:** بناء كل النوافذ والأهداف والمقاييس، واستبعاد العملات والفترة المحجوبة، وكتابة المجلد.
5. **Check:** إعادة قراءة الملف كما تفعل دفاتر التدريب وإعادة حساب بصمات كل مصفوفة.

بعد الانتهاء يمكنك تشغيل [`main.ipynb`](https://colab.research.google.com/github/yuosef772424/nig-timenet-torch/blob/main/notebooks/main.ipynb) (ضع في `ARTIFACT` المسار الناتج).

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/native-data-notebooks-uncertainty"
EXPECT_DEVICE = "cpu"                         # "cpu" أو "cuda": بناء البيانات لا يحتاج GPU؛ تشغيل cuda على جهاز CPU يفشل عمداً
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # مجلد عمل مؤقت (سجلات، أداة اعتماد git)؛ لا /tmp أبداً (E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = (("pyarrow", "pyarrow"), ("pandas_ta_classic", "pandas-ta-classic==0.8.32"), ("requests", "requests"))
RAW_ROOT = "/content/drive/MyDrive"            # يحوي history_1h/ وcrypto_data/asset_registry.csv (الشموع الخام)
# مجلد الخرج (العقد الناتج)
OUT_DIR = "/content/drive/MyDrive/crypto_model/preprocessed_data"
NAME = "preprocessing_output"                  # الملفان الناتجان: <NAME>_<ختم>.dataset و<NAME>_latest.dataset
PRESETS = ("hourly_w32_s8",)                   # رقع جاهزة من data/presets.py: hourly_w32_s8, hourly_4h_closed, dated_splits, tiny
CHANGES = {}                                   # مفاتيح منقَّطة فوق الرقع، مثل {"windows.stride": 4, "source.categories": ("all_global",)}
FETCH = ""                                     # "futures" أو "vision": نزّل أحدث الشموع أولاً؛ "" = قراءة فقط

## 1. Setup — تجهيز الجلسة
الخلية الأولى استيراد خاص: لا شيء من الحزمة موجود على الجهاز بعد، فهي تنزّل الملف الوحيد الذي لا يعتمد على شيء (`colab/bootstrap.py`) بتوكن GitHub وتنفّذه.
كل ما بعدها استيراد عادي: استنساخ نظيف للمستودع عند `TORCH_REF` وتثبيته مع `EXTRA_MODULES` (مكتبة المؤشرات الفنية `pandas-ta-classic` و`requests`)، ثم التحقق من الجهاز.
هنا `EXPECT_DEVICE = "cpu"` عن قصد: بناء البيانات لا يحتاج GPU.
**ما تتوقعه:** سطر commit ثم سطر torch/الجهاز. **فخ:** توكن منتهٍ يعطي `HTTP 401/404`؛ احذف `TOKEN_PATH` وأعد الخلية.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Settings — إعدادات البناء
`preset(...)` تطبّق الرقع المسمّاة بالترتيب ثم `CHANGES` فوقها. الافتراضيات هي إعداد الإنتاج للساعة (نافذة 32 وخطوة 8)؛ والرقع المتاحة:
`hourly_w32_s8` (الافتراضي بلا تغيير)، `hourly_4h_closed` (سياق 4h مغلق مع 1h)، `dated_splits` (تواريخ تقسيم صريحة وبداية للفترة المحجوبة)، `tiny` (بيانات تركيبية صغيرة للاختبار).
مفتاح غير معروف أو خاطئ الهجاء يفشل مع اقتراح، **قبل** بناء أي شيء. **ما تتوقعه:** طباعة إعدادات النوافذ وعدد العملات.

In [ ]:
from nig_timenet_torch.data.presets import preset
settings = preset(*PRESETS).updated({"source.root": RAW_ROOT, "output.directory": OUT_DIR, "output.name": NAME, "source.fetch": FETCH, **CHANGES})
coins = len(settings.source.coins) or settings.source.categories or "all"
print(settings.windows, coins)

## 3. Download — التنزيل (اختياري)
يعمل فقط حين تضبط `FETCH` على `"futures"` أو `"vision"`: يضيف أحدث شموع الساعة إلى الملفات الخام ثم يحدّث السجل. اتركه فارغاً (`""`) لقراءة الملفات كما هي.
**ما تتوقعه:** عدد الصفوف المضافة وعدد العملات. **فخ:** التنزيل يغيّر الملفات الخام نفسها على Drive، فيتغيّر العقد الناتج عن التشغيل السابق؛ لا تفعله في منتصف مقارنة.

In [ ]:
from nig_timenet_torch.data.fetch import fetch_all
from nig_timenet_torch.data.sources import resolve_coins
added = fetch_all(settings.source, resolve_coins(settings.source)) if FETCH else {}
print(sum(added.values()), "rows added for", len(added), "coins")

## 4. Build — البناء والكتابة
يبني كل نافذة وهدف ومقياس تماماً كما يفعل `crypto-signal-prediction` (الاختبارات تقارنها بت ببت)، ويستبعد العملات المحجوبة والفترة المحجوبة، ويكتب المجلد المؤرَّخ ونسخة `_latest`، ويطبع ملخصاً.
**كيف تقرأ المخرج:** الملخص يعدّ النوافذ والعملات والفريمات؛ وآخر سطر يذكر عدد الصفوف وأحدث وقت شمعة (`max bar time`) — يجب أن يسبق ختم الزمن. قد يستغرق هذا القسم وقتاً حسب عدد العملات.

In [ ]:
from nig_timenet_torch.data.pipeline import build_artifact, summary_lines
dataset, manifest = build_artifact(settings)
print("\n".join(summary_lines(dataset)))
print(manifest["export"]["rows"], "rows, max bar time", manifest["export"]["max_bar_time"])

## 5. Check — الفحص
يقرأ الملف الناتج بالطريقة نفسها التي تقرأه بها دفاتر التدريب، ويعيد حساب بصمة كل مصفوفة (`verify_hashes=True`).
**ما تتوقعه:** المسار والصيغة (`npy_dir` للمجلد) وعدد النوافذ. أي فشل هنا يعني أن الملف لا يصلح للتدريب؛ لا تستعمله.

In [ ]:
from nig_timenet_torch.data_io.loading import LoaderSettings, load_dataset
loaded = load_dataset(LoaderSettings(directory=OUT_DIR, name=NAME, verify_hashes=True))
print(loaded.path, loaded.format, len(loaded.dataset["last_candles"]), "windows")